# VietHandOCR Part 4: VietOCR Model Training

Welcome to **Part 3** of the VietHandOCR pipeline.
- **Previous Notebook**: [Part 3: Digital Image Processing (DIP) Pipeline](./03_Digital_Image_Processing.ipynb)
- **Next Notebook**: [Part 5: Evaluation & Inference](./05_Evaluation_and_Inference.ipynb)

## Introduction
Here we instantiate the `vgg_transformer` architecture using the VietOCR library. We load pre-trained weights to save massive amounts of compute time. The training loop integrates **Weights & Biases (W&B)** and collects **Out-of-Fold (OOF)** predictions.


In [ ]:
import os, gc, torch
import pandas as pd
import torch.nn as nn
import wandb

# Install VietOCR if not present
# !git clone https://github.com/pbcquoc/vietocr.git
# !pip install -q -e ./vietocr

from vietocr.tool.config import Cfg
from vietocr.model.trainer import Trainer
from vietocr.model.seqmodel.seq2seq import Seq2Seq
from vietocr.model.vocab import Vocab

def setup_vietocr_model():
    '''Loads the vgg_transformer config and builds the model.'''
    config = Cfg.load_config_from_name('vgg_transformer')
    config['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    
    # Customizing config for our dataset
    # config['dataset']['train'] = 'train_metadata.parquet'
    # config['dataset']['valid'] = 'val_metadata.parquet'
    
    # The build_model function or Trainer can be used. 
    # For custom MLOps (W&B, OOF), we often extract the model directly:
    # vocab = Vocab(config['vocab'])
    # model = Seq2Seq(vocab.size, **config['model'])
    # model.load_state_dict(torch.load('vgg_transformer.pth', map_location=config['device']))
    # model.to(config['device'])
    
    print("Model config loaded. Pre-trained weights prepared.")
    return config #, model


In [ ]:
# Training Loop Skeleton with MLOps (W&B, OOF)
wandb.init(project="VietHandOCR", name="resnet50-transformer-run1")
config = setup_vietocr_model()

# We use the built-in Trainer to build dataloaders
trainer = Trainer(config, pretrained=True)
trainer.config['trainer']['epochs'] = 20

vocab = Vocab(config['vocab'])
model = Seq2Seq(vocab.size, **config['model'])
model.to(config['device'])

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
scaler = torch.cuda.amp.GradScaler()

oof_predictions = []
train_dataloader = trainer.train_gen
val_dataloader = trainer.valid_gen

for epoch in range(20):
    model.train()
    for batch in train_dataloader:
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            loss = model(batch)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    
    val_loss = 0
    model.eval()
    with torch.no_grad():
        for val_batch in val_dataloader:
            preds = model(val_batch)
            oof_predictions.extend(preds)

    wandb.log({"epoch": epoch, "train_loss": loss.item(), "val_loss": val_loss})
    torch.cuda.empty_cache()
    gc.collect()

# Save OOF predictions
import pandas as pd
pd.DataFrame(oof_predictions).to_csv('oof_preds_vgg_transformer.txt', sep='\t', index=False, header=False)
torch.save(model.state_dict(), 'vietocr_finetuned.pth')
wandb.finish()